In [ ]:
# Import packages
import os
import pickle

import numpy as np 
import pandas as pd 

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from scipy import stats
import scipy.optimize as optimize 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfDaily.pkl")

In [ ]:
# NOTE VIX HV
target_VIX = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/target_VIX_hv.pkl")
predictions_random = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_random.pkl")
predictions_lr = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_lr.pkl")
predictions_lor = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_lor.pkl")
predictions_rfc = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_rfc.pkl")
predictions_mlc = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_mlc.pkl")

In [ ]:
# direction arrays
directions_VIX = target_VIX - target_VIX.shift()
directions_random = predictions_random  - predictions_random.shift()
directions_lor = predictions_lor
directions_rfc = predictions_rfc
directions_mlc = predictions_mlc

In [ ]:
# multiplicate element-wise, and count the positivie elements
def count_pos(series):
    count = 0
    for x in series:
        if x>0:
            count += 1
    return count

In [ ]:
direction_counts = [len(directions_VIX),0,0,0,0,0,0,0,0,0]
i=1
direction_counts[i] = count_pos(directions_random * directions_VIX);i+=1
direction_counts[i] = count_pos(directions_lor * directions_VIX);i+=1
direction_counts[i] = count_pos(directions_rfc * directions_VIX);i+=1
direction_counts[i] = count_pos(directions_mlc * directions_VIX);i+=1

# confusion matrix

In [ ]:
# directions_VIX=directions_VIX["2022-01-12":]
# directions_lr=directions_lr["2022-01-12":]
# directions_lor=directions_lor["2022-01-12":]
# directions_rf=directions_rf["2022-01-12":]
# directions_ml=directions_ml["2022-01-12":]

In [ ]:
print(len(directions_VIX))
print(len(directions_random))
print(len(directions_lor))
print(len(directions_rfc))
print(len(directions_mlc))

In [ ]:
directions_lor

In [ ]:
directions_VIX = directions_VIX[1:]
directions_random = directions_random[1:]
directions_lor = directions_lor[1:]
directions_rfc = directions_rfc[1:]
directions_mlc = directions_mlc[1:]

In [ ]:
print(len(directions_VIX))
print(len(directions_random))
print(len(directions_lor))
print(len(directions_rfc))
print(len(directions_mlc))

In [ ]:
vix = np.sign(directions_VIX)
ra = np.sign(directions_random)
lor = np.sign(directions_lor)
rfc = np.sign(directions_rfc)
mlc = np.sign(directions_mlc)

In [ ]:
vix = pd.DataFrame(data=vix)
ra = pd.DataFrame(data=ra)
lor = pd.DataFrame(data=lor)
rfc = pd.DataFrame(data=rfc)
mlc = pd.DataFrame(data=mlc)

In [ ]:
lor.dropna()

In [ ]:
vix.replace({1: "Up", -1:"Down",0:"Unkown"},inplace=True)
ra.replace({1: "Up", -1:"Down",0:"Unkown"},inplace=True)
lor.replace({1: "Up", -1:"Down",0:"Unkown"},inplace=True)
rfc.replace({1: "Up", -1:"Down",0:"Unkown"},inplace=True)
mlc.replace({1: "Up", -1:"Down",0:"Unkown"},inplace=True)

In [ ]:
import seaborn as sns
plt.rcParams['font.size'] = 13

def BenchmarkConfusionMatrix(y_true, y_pred, ModelName,ax):
    y_true=y_true.loc[y_true.index.isin(y_pred.index)]
    y_pred=y_pred.loc[y_pred.index.isin(y_true.index)]

    plt.rcParams['font.size'] = 16
    cf_matrix = confusion_matrix(y_true, y_pred, labels=["Up", "Down"])
#     cf_matrix = pd.DataFrame(data=cm, columns=['Actual Positive:Up', 'Actual Negative:Down'], 
#                                  index=['Predict Positive:Up', 'Predict Negative:Down'])
    group_names = ["True Neg","False Pos","False Neg","True Pos"]
    group_counts = ["{0:0.0f}".format(value) for value in
                    cf_matrix.flatten()]
    group_percentages1 = ["{0:.2%}".format(value) for value in
                        cf_matrix[0].flatten()/np.sum(cf_matrix[0])]
    group_percentages2 = ["{0:.2%}".format(value) for value in
                        cf_matrix[1].flatten()/np.sum(cf_matrix[1])]
    group_percentages = group_percentages1 + group_percentages2                     
    labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
            zip(group_names,group_counts,group_percentages)]
    labels = np.asarray(labels).reshape(2,2)

    disp = sns.heatmap(pd.DataFrame(data=cf_matrix,columns=["Up", "Down"],index=["Up", "Down"])
                                    , annot=labels, fmt='', vmin=0, vmax=39, cmap='OrRd',ax=ax)#'Blues')
    disp.set_title(ModelName + " Vs Actual", fontweight="bold")
    disp.xaxis.labelpad = 10

    disp.set_xlabel('Predicted Values\n')

    disp.set_ylabel('Actual Values')

In [ ]:
sns.set(font_scale=1.5)

fig, axes = plt.subplots(2, 2, figsize=(12,12), dpi= 80)
i=0;j=0
BenchmarkConfusionMatrix(vix,ra, "Random-Sample",axes[i,j]);j=j+1
BenchmarkConfusionMatrix(vix,lor, "Logistic-Regression",axes[i,j]);j=0;i=1
BenchmarkConfusionMatrix(vix,rfc,"Random-Forest-Classifier",axes[i,j]);j=j+1
BenchmarkConfusionMatrix(vix,mlc,"Machine-Learning-Classifier",axes[i,j])
plt.subplots_adjust(top=0.95)
plt.suptitle("Classification-Models\nDirection-Forecasting\nPress-Release\n", fontsize=30, fontweight="bold")
plt.tight_layout(pad=0.5)

plt.show()

In [ ]:
directions_sentiment = count_pos(df["Sentiment return"].values * df["VIX return"].values)